# Fake account detection

This notebook uses the same validated training pipeline as the website. Run it from the project folder using the project Python environment. Rebuilding updates all model files and dashboard metrics together.


In [ ]:
from pathlib import Path
import json
import pickle
import pandas as pd
from train_model import main, BASE_DIR
from features import FEATURES, validate


In [ ]:
data = pd.read_csv(BASE_DIR / 'data.csv')
print('Original shape:', data.shape)
print('Exact duplicate rows:', data.duplicated().sum())
print('Labels:', data['fake'].value_counts().to_dict())
print('Missing values:', data.isna().sum().to_dict())
data.head()

## Train and evaluate

The shared pipeline validates input, removes exact duplicates, uses a stratified split with seed 42, scales Logistic Regression inputs, and saves models plus evaluation metrics.


In [ ]:
main()
metrics = json.loads((BASE_DIR / 'model_metrics.json').read_text())
pd.DataFrame(metrics['models']).T[['accuracy', 'precision', 'recall', 'f1']]

In [ ]:
for name, scores in metrics['models'].items():
    print(name)
    print(pd.DataFrame(scores['confusion'], index=['Actual genuine', 'Actual fake'], columns=['Predicted genuine', 'Predicted fake']))

In [ ]:
pd.DataFrame(metrics['importance'], columns=['Feature', 'Random Forest importance'])

## Try a prediction

Inputs stay numeric and retain the training feature names. Confidence is uncalibrated and does not verify identity.


In [ ]:
with (BASE_DIR / 'Random_forest.sav').open('rb') as handle:
    model = pickle.load(handle)
sample = validate(pd.DataFrame([[1, 0, 1, 0, 0, 0, 0, 0, 114, 490, 1093]], columns=FEATURES))
print('Class:', int(model.predict(sample)[0]))
print('Probabilities:', dict(zip(model.classes_.tolist(), model.predict_proba(sample)[0].tolist())))